In [2]:
import requests as rq
import bs4

In [40]:
import json
import re


def get_eg_link(title):
    params = {
        "sort": "score,DESC",
        "page": 0,
        "size": 10,
        "query": f'"{title}"',
        "f.dateIssued": "[2007 TO *],equals",
        "embed": "thumbnail, item/thumbnail",
    }
    r = rq.get(
        "https://diglib.eg.org/server/api/discover/search/objects",
        params,
    ).json()
    res_list = r["_embedded"]["searchResult"]["_embedded"]["objects"]
    res_object = res_list[0] if len(res_list) > 0 else None
    # print(json.dumps(res_object, indent=2))
    if res_object is None:
        return {}
    link = f"https://diglib.eg.org/items/{res_object['_embedded']['indexableObject']['id']}"
    doi = res_object["_embedded"]["indexableObject"]["handle"]
    authors = res_object["_embedded"]["indexableObject"]["metadata"][
        "dc.contributor.author"
    ]
    abstr = res_object["_embedded"]["indexableObject"]["metadata"].get(
        "dc.description.abstract", []
    )
    thumbnail = res_object["_embedded"]["indexableObject"]["_embedded"]["thumbnail"]["_links"]["content"]["href"]
    author_names = [a["value"] for a in authors]
    return {
        "link": link,
        "doi": doi,
        "authors": author_names,
        "abstr": abstr,
        "thumbnail": thumbnail
    }


resp = get_eg_link("Parameter Space Analysis through Guided Visual Interpolations")

In [3]:
program_html=rq.get("https://conferences.eg.org/vcbm2026/full-program/").text


In [4]:
soup = bs4.BeautifulSoup(program_html, "html.parser")

In [45]:
entries = list(soup.find("div", class_="entry-content").children)
day = ""
timeslot = ""
import dataclasses


@dataclasses.dataclass
class Paper:
    day: str
    timeslot: str
    title: str
    link: str
    authors: str
    doi: str = ""
    thumbnail: str = ""


papers = []
for entry in entries:
    if entry.name == "h3":
        day = entry.text
    elif entry.name == "h5":
        timeslot = entry.text
    elif entry.name == "div":
        talks = entry.find_all("p")
        for talk in talks:
            if talk.find("strong") is None:
                continue
            name = talk.find("strong").text
            link = talk.find("a")["href"] if talk.find("a") is not None else ""
            authors = talk.find("em").text if talk.find("em") is not None else ""
            paper = Paper(
                day=day, timeslot=timeslot, title=name, link=link, authors=authors
            )
            papers.append(paper)
            print(f"{day} | {timeslot} | {name} | {link} | {authors}")

Wednesday, September 16 | 16:30 onwards • Poster & Demo Session with Welcome Reception | Interactive Visualization of Dynamic Functional Connectivity in the ABIDE Autism Dataset | https://conferences.eg.org/vcbm2026/wp-content/uploads/sites/27/2026/09/Schielen_ABIDE.pdf | Daniel Eliad, Sjir J.C. Schielen, Albert P. Aldenkamp, Danny Ruijters, Svitlana Zinger
Wednesday, September 16 | 16:30 onwards • Poster & Demo Session with Welcome Reception | Effective Guidance in Immersive, Simulation-based Ophthalmology Education | https://conferences.eg.org/vcbm2026/wp-content/uploads/sites/27/2026/09/Zaitseva_GuidanceVR.pdf | Mariia Zaitseva, Doğukan Zal, Peter Walter, Torsten Kuhlen, Tim Gerrits
Wednesday, September 16 | 16:30 onwards • Poster & Demo Session with Welcome Reception | MindJourneys: An Interactive Narrative Visualization of Depression Case Reports | https://conferences.eg.org/vcbm2026/wp-content/uploads/sites/27/2026/09/Swadia_MindJourneys.pdf | Dhruvi Gaurav Swadia, Nivedita Kowla

In [46]:
by_day: dict[str, list[Paper]] = {}
for paper in papers:
    if paper.day not in by_day:
        by_day[paper.day] = []
    by_day[paper.day].append(paper)


In [47]:
for day, papers in by_day.items():
    print(f"## {day}")
    for paper in papers:
        if paper.link != "":
            continue
        print(f"Getting EG link for {paper.title}")
        eg_info = get_eg_link(paper.title)
        paper.link = eg_info.get("link", "")
        paper.doi = eg_info.get("doi", "")
        paper.authors = ", ".join(eg_info.get("authors", []))
        paper.thumbnail = eg_info.get("thumbnail", "")
        print(f"Found EG link: {paper.link}, DOI: {paper.doi}")

## Wednesday, September 16
## Thursday, September 17
Getting EG link for ConGAT: Context-Aware Graph Attention Visual Analysis for 3D Region of Interest Discovery in Multiplexed Microscopy Images
Found EG link: , DOI: 
Getting EG link for MultiPASE: Shape-Aware Comparative 3D Genome Analysis
Found EG link: https://diglib.eg.org/items/8e0ab2b4-c6af-45e0-bcbd-5fb00b37fbb5, DOI: 10.2312/vcbm20261015
Getting EG link for Exploring Genes in the Brain Through Multimodal Views Driven by Disorder Ontologies
Found EG link: https://diglib.eg.org/items/6282f7d7-caea-4a4e-ab2d-5aa537146d33, DOI: 10.2312/vcbm20261001
Getting EG link for Interactive 3D Exploration of Large-Scale Neuronal Network Simulations with a Graph Lens
Found EG link: https://diglib.eg.org/items/94fb9f50-d78b-4aa1-bf56-4601a570fe4c, DOI: 10.2312/vcbm20261012
Getting EG link for An End-to-end Framework for Quantifying the Exploratory Behavior of Multicolumnar Neurons in Drosophila
Found EG link: https://diglib.eg.org/items/5a3a17

In [53]:
for i, (day, papers) in enumerate(by_day.items()):
    with open(f"g_day_{i + 2}.md", "w") as f:
        f.write(f"""---
title: VCBM Day {i + 2}
date: 2026-09-{i + 16}
---
# {day}
""")
        for paper in papers:
            f.write(f"""## {paper.title}
""")        
            if paper.thumbnail != "":
                f.write(f"\n![{paper.thumbnail}]({paper.thumbnail})\n\n")
            if paper.authors != "":
                f.write(f"_Authors: {paper.authors}_\n")
            if paper.link != "":
                f.write(f"- Link: [EG Digilib]({paper.link})\n")
            if paper.doi != "":
                f.write(f"- DOI: [{paper.doi}](https://doi.org/{paper.doi})\n")


In [19]:
len(entries)

77

In [34]:
entries[5].name

'h3'

In [14]:
from pathlib import Path


root_path = Path(".")
mds = root_path.glob("day_*._md")
mds = list(mds)
mds

[PosixPath('day_5._md'),
 PosixPath('day_4._md'),
 PosixPath('day_3._md'),
 PosixPath('day_2._md'),
 PosixPath('day_1._md')]

In [16]:
resp

{'link': 'https://diglib.eg.org/items/860c1d60-ddda-49ba-93b2-21f285c4c410',
 'doi': '10.2312/mlvis20261001',
 'authors': ['Kantz, Benedikt',
  'Waldert, Peter',
  'Lengauer, Stefan',
  'Staudinger, Clemens',
  'Schuster, Stefan',
  'Schreck, Tobias'],
 'abstr': [{'value': 'We propose Parameter Space Analysis through Guided Visual Interpolations (ParamInter), a novel tool for high dimensional input parameter space analysis by making interpolation towards optimal parameter sets explorable using guided analytics. The interpolation is accompanied by both small multiples in linked views, and utilizes t-Distributed Stochastic Neighbor Embedding (t-SNE) representations to show an interpolation overview. ParamInter uses a guided exploration loop focusing on the interpolation towards user-specified target parameters from many output parameters. The exploration process is additionally guided through eXplainable Artificial Intelligence (XAI)-based effect suggestions throughout our tool. ParamInt

In [17]:
out_path = root_path
out_path.mkdir(exist_ok=True)
for md in mds:
    out_file = out_path / f"ev_{md.stem}.md"
    processed_content = []
    with open(md, "r") as f:
        content = f.read()
    lines = content.splitlines()
    for line in lines:
        processed_content.append(line)
        if line.startswith("## "):
            title = line[3:]
            print(f"Processing {title}")
            eg_info = get_eg_link(title)
            author_text = (
                ", ".join(eg_info["authors"])
                if "authors" in eg_info
                else "No authors found"
            )
            authors = (
                f"> Authors: {author_text}"
                if "authors" in eg_info
                else "No authors found"
            )
            link = (
                f"\n[EG Link]({eg_info['link']})"
                if "link" in eg_info
                else "No EG link found"
            )
            thumbnail = (
                f"\n![Thumbnail]({eg_info['thumbnail']})"
                if "thumbnail" in eg_info
                else ""
            )
            processed_content.append(authors)
            processed_content.append(link)
            processed_content.append(thumbnail)
    with open(out_file, "w") as f:
        f.write("\n".join(processed_content))

Processing Can LLMs Simulate Target Users in Visualization Case Studies?
Processing Beauty in the Eye of AI: Aligning LLMs and Vision Models with Human Aesthetics in Network Visualization
Processing Do Graph Drawing Aesthetics Matter for AI? A Replication of Foundational Studies in Graph Readability
Processing How Do LLMs See Charts? A Comparative Study on High-Level Visualization Comprehension in Humans and LLMs
Processing Uncertainty-Aware Visual Analysis of Force Networks in 2D Granular Materials
Processing Uncertainty Visualization for Biomolecular Structures: An Empirical Evaluation
Processing GraphHeatY: Graph-Centered Visual Analysis for Building Design
Processing GEVIS: A Workflow-Driven Visual Analytics Approach to Differential Gene Expression Analysis
Processing SemiConLens : Visual Analytics for 2D Semiconductor Discovery
Processing A Scalable System for Visual Analysis of Ocean Data
Processing Class Angular Distortion Index for Dimensionality Reduction
Processing SPINE: VAE

In [18]:
import sys
sys.path.append('../../../processing')

In [19]:
from ocr import extract_poster_img, image_to_text

In [20]:
import tqdm
import cv2
poster_path = Path("images/photos/posters")
warped_path = poster_path / "warped"
warped_path.mkdir(exist_ok=True)
imgs = list(poster_path.glob("*.jpeg"))
print(imgs)
# for img in tqdm(imgs):
#     print(img)
#     img_dat=cv2.imread(img)
#     final=scan(img_dat)
#     cv2.imwrite('out/'+img.split('/')[-1], final)
len(imgs)
for img in tqdm.tqdm(imgs):
    warped = extract_poster_img(str(img))
    target_path = warped_path / img.name.replace(".jpeg", "_warped.jpeg")

    cv2.imwrite(str(target_path), warped)

[PosixPath('images/photos/posters/IMG_7154.jpeg'), PosixPath('images/photos/posters/IMG_7152.jpeg'), PosixPath('images/photos/posters/IMG_7153.jpeg')]


100%|██████████| 3/3 [00:02<00:00,  1.10it/s]


In [21]:
from utils import authors_text, biggest_text
import pandas as pd

In [22]:
results = []
warped_imgs = list(warped_path.glob("*.jpeg"))
for img in tqdm.tqdm(warped_imgs):
    text = image_to_text(str(img))
    authors = authors_text(text)
    title = biggest_text(text)
    results.append({
        "title": title,
        "authors": authors,
        "image_path": img
    })
results_df = pd.DataFrame(results)

100%|██████████| 3/3 [16:49<00:00, 336.55s/it]


In [23]:
results_df

,title,authors,image_path
0,"orall and subset-specific accuracy, (2) stacke...","B. T. Arnold, B. Greiner, C. Lange, and C. Faß...",images/photos/posters/warped/IMG_7153_warped.jpeg
1,When topic modeling is introduced into the mic...,"3) COMER L, HUO P. COLLELUORIC. ZHAOH. AKRAM M...",images/photos/posters/warped/IMG_7152_warped.jpeg
2,Exploring Trends and Functional Traits for Com...,"I. Nijns, P. Vanormelingen, W. Veraghtert, P. ...",images/photos/posters/warped/IMG_7154_warped.jpeg


In [24]:
out_md_path = root_path / "ev_poster_links.md"
with open(out_md_path, "w") as f:
    for _, row in results_df.iterrows():
        f.write(f"## {row['title']}\n")
        f.write(f"> Authors: {row['authors']}\n")
        f.write(f"![Poster Image]({row['image_path']})\n\n")